In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
import re

from sklearn import model_selection, preprocessing, linear_model, metrics
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from nltk.tokenize import RegexpTokenizer

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from textblob import Word
nltk.download('wordnet')

from termcolor import colored
from warnings import filterwarnings
filterwarnings('ignore')

from sklearn import set_config
set_config(print_changed_only = False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [9]:
import numpy as np
import pandas as pd
import os

from google.colab import files

uploaded = files.upload()

Saving ikn-tanyarlfes.csv to ikn-tanyarlfes (1).csv


In [10]:
data = pd.read_csv('ikn-tanyarlfes.csv')
data['tweet'] = data['tweet'].astype(str)

In [11]:
for id_str in data['tweet'].head():
    print(id_str, type(id_str))

@tanyarlfes 14M buat satu rumah mentri padahal rumah 400-500 juta udah bagus 14M tuh bisa ngasih makan berapa juta jiwa cobaa wkwk zolim bgt sumpah dari awal gak setuju bgt sama proyek ini mana ngusir masyarakat adat juga (cmiiw yaa aku baca di portal berita) <class 'str'>
@lasakna @tanyarlfes dah gile ya anjir 14M buat 1 rumah pejabat wkwk mending luhk kasih akses pendidikan daerah pelosok noh biar ga menuju indonesia (c)emas  <class 'str'>
@tanyarlfes beritanya disini sebenernya beberapa portal berita juga bahas ini tapi aku cari di historyku udh gaadaa:( aku open discussion yaa (gak nerima diskusi sama buzz3r malesss) https://t.co/8KPtbyuSg3 <class 'str'>
@lasakna @tanyarlfes 14M memang mahal banget tapi standarnya nggak 400-500 juta juga. Harga 500 untuk sebuah rumah utuh masih kurang minimal kisaran brp M. Sanggah dikit aja <class 'str'>
@lasakna @tanyarlfes Anggarannya 14M???? Per 1 rumah pejabat. Padahal dg 14M udh brp banyak masyarakat indo yg bisa disejahterakan <class 'str'>


## **Preprocessing**

In [12]:
print("Total data duplikat {}".format(data.duplicated().sum()))

Total data duplikat 0


In [13]:
print("Total missing values\n{}".format(data.isnull().sum()))

Total missing values
tweet         0
username      2
anotator 1    0
anotator 2    0
anotator 3    0
labeling      0
dtype: int64


In [14]:
data.dropna(inplace=True)

In [15]:
print("Total missing values\n{}".format(data.isnull().sum()))

Total missing values
tweet         0
username      0
anotator 1    0
anotator 2    0
anotator 3    0
labeling      0
dtype: int64


In [16]:
data

,tweet,username,anotator 1,anotator 2,anotator 3,labeling
0,@tanyarlfes 14M buat satu rumah mentri padahal...,lasakna,negatif,negatif,negatif,negatif
1,@lasakna @tanyarlfes dah gile ya anjir 14M bua...,soizleil,negatif,negatif,negatif,negatif
2,@tanyarlfes beritanya disini sebenernya bebera...,lasakna,positif,netral,netral,netral
3,@lasakna @tanyarlfes 14M memang mahal banget t...,serebeloom,negatif,negatif,positif,negatif
4,@lasakna @tanyarlfes Anggarannya 14M???? Per 1...,louusun,negatif,negatif,negatif,negatif
...,...,...,...,...,...,...
84,@tanyarlfes Niat biar org jakarta mutasi ke ik...,qwertypnsjje,negatif,negatif ‍️,negatif,negatif
85,@tanyarlfes Menurutku ikn itu ga bisa dipriori...,Newbie102000,negatif,negatif ‍️,negatif,negatif
86,@tanyarlfes Pernah baca jepang juga pernah ber...,MasamuneMasamu2,negatif,negatif ‍️,negatif,negatif
87,@tanyarlfes Terlalu maksain diri kyk kaum mene...,AQuiscent,negatif,negatif ‍️,negatif,negatif


### Convert huruf uppercase ke lowercase

In [17]:
data['tweet'] = data['tweet'].apply(lambda x: " ".join(x.lower() for x in x.split()))

In [18]:
data['tweet']

,tweet
0,@tanyarlfes 14m buat satu rumah mentri padahal...
1,@lasakna @tanyarlfes dah gile ya anjir 14m bua...
2,@tanyarlfes beritanya disini sebenernya bebera...
3,@lasakna @tanyarlfes 14m memang mahal banget t...
4,@lasakna @tanyarlfes anggarannya 14m???? per 1...
...,...
84,@tanyarlfes niat biar org jakarta mutasi ke ik...
85,@tanyarlfes menurutku ikn itu ga bisa dipriori...
86,@tanyarlfes pernah baca jepang juga pernah ber...
87,@tanyarlfes terlalu maksain diri kyk kaum mene...


### Normalisasi Teks

In [19]:
norms = {
    'bgt': 'banget', 'udh': 'sudah', 'dg': 'dengan', 'brp': 'berapa',
    'yg': 'yang', 'blm': 'belum', 'mon': 'mohon', 'sprt': 'seperti', 'lbh': 'lebih',
    'tp': 'tapi', 'emg': 'emang', 'pdhl': 'padahal', 'jg': 'juga', 'bkn': 'bukan',
    'krn': 'karena', 'msih': 'masih', 'skrg': 'sekarang', 'dn': 'dan',
    'sbnyk': 'sebanyak', 'cmn': 'cuman', 'pda': 'pada', 'knp': 'kenapa', 'kyk': 'kayak',
    'orng': 'orang', 'ak': 'aku', 'dr': 'dari', 'kmrn': 'kemarin', 'bpn': 'balikpapan',
    'ikam': 'ikan', 'karna': 'karena', 'uda': 'sudah', 'msi': 'masih', 'lebi': 'lebih',
    'mhs': 'mahasiswa', 'lg': 'lagi', 'scr': 'secara', 'cttn': 'catatan', 'gw': 'gue',
    'jd': 'jadi', 'sbagai': 'sebagai', 'dpt': 'dapat', 'bwt': 'buat', 'kl': 'kalau',
    'sebnernya': 'sebenarnya'
}

def normalisasi(str_text):
    words = str_text.split()  # Membagi teks menjadi kata-kata
    normalized_words = []
    for word in words:
        if word.lower() in norms:  # Mengecek kata dalam norms, tidak memperhatikan huruf besar/kecil
            normalized_words.append(norms[word.lower()])  # Mengganti kata jika ditemukan dalam norms
        else:
            normalized_words.append(word)  # Menambahkan kata tanpa perubahan
    return ' '.join(normalized_words)

In [20]:
data['tweet'] = data['tweet'].apply(lambda x: normalisasi(x))
print(data['tweet'])

0     @tanyarlfes 14m buat satu rumah mentri padahal...
1     @lasakna @tanyarlfes dah gile ya anjir 14m bua...
2     @tanyarlfes beritanya disini sebenernya bebera...
3     @lasakna @tanyarlfes 14m memang mahal banget t...
4     @lasakna @tanyarlfes anggarannya 14m???? per 1...
                            ...                        
84    @tanyarlfes niat biar org jakarta mutasi ke ik...
85    @tanyarlfes menurutku ikn itu ga bisa dipriori...
86    @tanyarlfes pernah baca jepang juga pernah ber...
87    @tanyarlfes terlalu maksain diri kayak kaum me...
88    @tanyarlfes sebenarnya tujuan awalnya buat pem...
Name: tweet, Length: 87, dtype: object


### Menghapus tanda baca dan username yang tertaut

In [21]:
data['tweet'] = data['tweet'].str.replace(r'[:/\\\-?.()!,\-~]+', '', regex=True)

In [22]:
data['tweet'] = data['tweet'].str.replace('@tanyarlfes', '').str.replace('@lasakna', '')

In [23]:
data['tweet']

,tweet
0,14m buat satu rumah mentri padahal rumah 4005...
1,dah gile ya anjir 14m buat 1 rumah pejabat w...
2,beritanya disini sebenernya beberapa portal b...
3,14m memang mahal banget tapi standarnya ngga...
4,anggarannya 14m per 1 rumah pejabat padahal ...
...,...
84,niat biar org jakarta mutasi ke ikn tapi feel...
85,menurutku ikn itu ga bisa diprioritaskan tapi...
86,pernah baca jepang juga pernah berpikiran mem...
87,terlalu maksain diri kayak kaum menengah yang...


### Menghapus angka

In [24]:
def cleaning_numbers(data):
    return re.sub('[0-9]+', '', data)

In [25]:
data['tweet'] = data['tweet'].apply(lambda x: cleaning_numbers(x))
data['tweet'].tail(15)

,tweet
73,prioritas ikn itu urgensinya ga mendesak di i...
74,dzolim lu pemerintah bangun ikn sebagai tanda...
75,ibu kota negara di pindahkan ke daerah yang b...
76,g bermanfaat g ngaruh buat orang kaltim gue s...
77,saya sebagai orang awam ikn tidak akan pernah...
78,banyak cara kalau mau pemerataan banyak hal y...
79,masih byk daerah yang deserve dapat perbaikan...
81,gue setuju banget masalah pemerataan biar gk ...
82,menurutku sih ikn ga penting banget klo mau p...
83,buat yang bilang ikn bagus untuk pemerataan ...


### Menghapus stopwords

In [26]:
sw = stopwords.words("indonesian")
data['tweet'] = data['tweet'].apply(lambda x: " ".join(x for x in x.split() if x not in sw))

### Tokenisasi

In [27]:
tokenizer = RegexpTokenizer(r'\w+')
data['tweet'] = data['tweet'].apply(tokenizer.tokenize)

In [28]:
data['tweet'].head()

,tweet
0,"[m, rumah, mentri, rumah, juta, udah, bagus, m..."
1,"[dah, gile, ya, anjir, m, rumah, pejabat, wkwk..."
2,"[beritanya, sebenernya, portal, berita, bahas,..."
3,"[m, mahal, banget, standarnya, nggak, juta, ha..."
4,"[anggarannya, m, rumah, pejabat, m, masyarakat..."


### Stemming

In [29]:
st = nltk.PorterStemmer()
def stemming_on_text(data):
    text = [st.stem(word) for word in data]
    return text  # FIX: sebelumnya 'return data' sehingga hasil stemming tidak pernah dipakai

data['tweet']= data['tweet'].apply(lambda x: stemming_on_text(x))


In [30]:
data['tweet'].head()

,tweet
0,"[m, rumah, mentri, rumah, juta, udah, bagu, m,..."
1,"[dah, gile, ya, anjir, m, rumah, pejabat, wkwk..."
2,"[beritanya, sebenernya, portal, berita, baha, ..."
3,"[m, mahal, banget, standarnya, nggak, juta, ha..."
4,"[anggarannya, m, rumah, pejabat, m, masyarakat..."


## **Split Dataset**

In [31]:
X = data['tweet']
y = data['labeling']

In [32]:
from sklearn.model_selection import train_test_split

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(data['tweet'], data['labeling'], test_size=0.2, random_state=42)

## **TF-IDF**

In [33]:
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, lowercase=False)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

## **Mengatasi Data yang tidak imbang dengan SMOTE**

In [34]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(k_neighbors=3, random_state=43)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

# **Model**

### SVC

In [35]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

model = SVC()
model.fit(X_train_smote, y_train_smote)
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9444444444444444


### Logistic Regression

In [36]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_smote, y_train_smote)
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8888888888888888


### Stochastic Gradient Descent

In [37]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier()
model.fit(X_train_smote, y_train_smote)
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9444444444444444


### Ensemble Stacking Classifier dengan SVC, Decision Tree, dan Logistic Regression

In [38]:
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

model1 = SVC(random_state=42)
model2 = DecisionTreeClassifier(random_state=0)
model3 = LogisticRegression(random_state=0)

ensembleSC = StackingClassifier(estimators=[('svc', model1), ('dt', model2), ('lr', model3)], final_estimator=SVC())
ensembleSC.fit(X_train_smote, y_train_smote)

y_pred_ensemble = ensembleSC.predict(X_test_tfidf)

print(f"Akurasi prediksi: {accuracy_score(y_test, y_pred_ensemble)}")


Akurasi prediksi: 0.9444444444444444


### Random Forest

In [39]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier()
model_rf.fit(X_train_smote, y_train_smote)
y_pred_rf = model_rf.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.9444444444444444
